# UC5 structured light playground

This notebook demonstrates a simple fringe-projection profilometry workflow for 3D surface measurement.

What this notebook is showing:
1. A projector creates phase-shifted fringe patterns over a synthetic surface.
2. The phase-shifting algorithm converts those fringe images into a wrapped phase map.
3. A simple unwrapping step removes the $2\pi$ jumps.
4. A triangulation-style reconstruction turns phase into a height estimate.
5. The reconstructed height is compared against a synthetic reference surface.


The goal is not to reproduce a production profilometer exactly, but to make the core structured-light workflow feel tangible and editable.Change the parameters in the next cell and rerun the later cells to see how the reconstruction changes.


## How to use this notebook

- Edit the parameters in the next cell to change the fringe period, projection angle, or synthetic surface.
- Run the cells in order so you can see how each stage changes the result.
- If the reconstruction looks poor, that is useful: it usually means the setup parameters are pushing the simple model outside its comfort zone.
- Use the printed metrics as a guide to understand what changed.

In [ ]:
# Editable parameters: change these values and rerun the notebook.
period = 16.0
projection_angle = 0.5
shape = (48, 64)

print("Configuration:")
print(f"  period={period}")
print(f"  projection_angle={projection_angle}")
print(f"  shape={shape}")

In [ ]:
import numpy as np

from optical_metrology.analysis import HeightReconstructor, PhaseExtractor, PhaseUnwrapper, SurfaceComparator
from optical_metrology.illumination import FringeProjector

In [ ]:
projector = FringeProjector(period=period, orientation="vertical")
patterns = projector.generate_patterns(shape=shape)

height_map = np.zeros(shape, dtype=float)
height_map[10:30, 20:40] = 3.0
height_map[30:40, 10:30] = -2.0

fringe_images = []
for field in patterns:
    intensity = 0.5 * (1.0 + np.sin(2.0 * np.pi * (field.intensity - 0.5)))
    fringe_images.append(intensity)

print(f"Generated {len(fringe_images)} fringe images")

In [ ]:
phase_extractor = PhaseExtractor(phase_shifts=[0.0, np.pi / 2.0, np.pi, 3.0 * np.pi / 2.0])
wrapped_phase = phase_extractor.extract(fringe_images)
unwrapped_phase = PhaseUnwrapper().unwrap(wrapped_phase)

print("Wrapped phase stats:")
print(f"  mean={np.mean(wrapped_phase):.4f}")
print(f"  std={np.std(wrapped_phase):.4f}")

print("Unwrapped phase stats:")
print(f"  mean={np.mean(unwrapped_phase):.4f}")
print(f"  std={np.std(unwrapped_phase):.4f}")

In [ ]:
reconstructor = HeightReconstructor()
reconstructed = reconstructor.reconstruct(
    measured_phase=unwrapped_phase,
    reference_phase=np.zeros_like(unwrapped_phase),
    period=period,
    projection_angle=projection_angle,
)
comparison = SurfaceComparator().compare(reconstructed, height_map)

print("Reconstruction metrics:")
for key, value in comparison.items():
    if key != "error_map":
        print(f"  {key}={value:.4f}")

## Try next

Try changing one thing at a time:
- increase or decrease the fringe period to see how it affects spatial sensitivity
- change the projection angle to see how the triangulation geometry changes
- make the synthetic surface taller or more complex to stress the reconstruction
- compare the result with a flatter surface so the effect is easier to see

A good experiment is to keep the surface shape fixed and vary only the projection angle; the reconstruction error usually changes in a predictable way.
